# SEA-Spoof — สำรวจ metadata จาก Hugging Face ครบสาม split
**ขอบเขต:** อ่านต้นทางออนไลน์ครั้งแรก ไม่ใช้ Parquet/manifest/เสียง SEA-Spoof เดิมในเครื่อง ไม่เลือกคอลัมน์ `audio` และไม่โหลดไฟล์เสียงทั้งชุด

อ่าน metadata ทุกภาษาเพื่อดูภาพรวม แล้ววิเคราะห์ **ภาษาไทย (`language = th`)** จาก Train / Validation / Evaluation โดยรักษา split เดิม

เริ่มอ่านหัวข้อ 1–4 เพื่อดูจำนวนคอลัมน์/คลิปและ missing data ก่อนรายละเอียด ไม่แสดงตาราง missing ซ้ำอีกชุด

**สิทธิ์:** ต้องใช้บัญชี Hugging Face ที่ได้รับอนุมัติ SEA-Spoof แล้ว ห้ามใส่ token ใน notebook หรือเผยแพร่ metadata รายแถวกับผู้ไม่มีสิทธิ์ ผลทั้งหมดอยู่ในโฟลเดอร์ที่ Git ignore

[Dataset card](https://huggingface.co/datasets/Jack-ppkdczgx/SEA-Spoof) | Revision `132f5dca9b6efe39cf1d3b54a858f167f5a421fc`

Notebook นี้ออกแบบสำหรับ repo ใน VS Code + kernel `.venv` หากใช้ Colab ต้องนำ helper ไปด้วยและตั้ง project root ให้ถูก ไม่อ่าน path D: จาก Colab โดยตรง

## 1. ตั้งค่าและขอบเขต
ติดตั้งด้วย `python -m pip install -e ".[eda]"` จาก project root และ login Hugging Face ผ่าน `hf auth login` แยกจาก notebook อย่าเขียน token ลง cell

ค่า `False`, `0`, และข้อความ `none` ไม่ถือเป็น missing โดยอัตโนมัติ

In [ ]:
# ตั้งค่า — ไม่อ่าน dataset เดิมในเครื่อง
import json
import sys
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").is_file()
             and (p / "data/exploration/sea_spoof/sea_spoof_metadata_utils.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("เปิด notebook จาก project root หรือโฟลเดอร์ด้านใน และต้องมี helper")
sys.path.insert(0, str(ROOT))
from data.exploration.sea_spoof.sea_spoof_metadata_utils import (
    DATASET, DEFAULT_REVISION, SPLITS, READ_COLUMNS, MEANINGS,
    fetch_metadata, blank_mask, missing_profile, normalized_text,
    overlap_table, namespaced_voice_keys,
)
OUTPUT = ROOT / "data/exploration/sea_spoof/outputs/metadata_only"
OUTPUT.mkdir(parents=True, exist_ok=True)
REVISION = DEFAULT_REVISION
STARTED = datetime.now(timezone.utc).isoformat()
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_colwidth", 90)
tables = {}

def table(name, frame, limit=30):
    tables[name] = frame
    frame.to_csv(OUTPUT / f"{name}.csv", index=False, encoding="utf-8-sig")
    print(f"{name}: {len(frame):,} rows" + (f" (แสดง {limit} แถวแรก)" if len(frame) > limit else ""))
    display(frame.head(limit))
    return frame

def save_plot(name):
    plt.tight_layout()
    plt.savefig(OUTPUT / f"{name}.png", bbox_inches="tight")
    plt.show()

print("Project root:", ROOT)
print("Revision:", REVISION)
print("Python:", sys.executable)
print("Scope: metadata only; no audio; no original local dataset files")

## 2. อ่าน metadata จากต้นทาง
ครั้งแรกอ่านจาก HF ด้วย HTTP byte ranges: Parquet footer/schema และคอลัมน์ metadata เท่านั้น รวมทั้ง Train ที่ยังไม่เคยโหลดเสียงมา

จะบันทึก **snapshot metadata ใหม่** ที่ `data/raw/sea_spoof_metadata/<revision>/` ครั้งถัดไปใช้เฉพาะ snapshot ใหม่นี้หลังตรวจ SHA-256 ไม่ใช้ข้อมูล SEA-Spoof เดิม

ไม่ได้ดาวน์โหลดทั้ง shard หรือบันทึกเสียง แต่การอ่าน footer อาจมี bytes ส่วนท้ายปะปน จึงไม่อ้างว่า network รับ bytes ที่เกี่ยวกับเสียงเป็นศูนย์อย่างเคร่งครัด

In [ ]:
# ดึงต้นทางออนไลน์ครั้งแรก; ไม่ใช้ sea_spoof_hf / manifests / processed audio
snapshot, provenance = fetch_metadata(ROOT, REVISION, workers=4)
metadata = pd.read_parquet(snapshot)
assert "audio" not in metadata.columns
assert set(metadata["official_split"]) == set(SPLITS)
assert metadata["split"].eq(metadata["official_split"]).all()
thai = metadata.loc[metadata["language"].astype("string").str.strip().eq("th")].copy()
assert len(thai) > 0
source_columns = provenance["source_columns"]

quick = pd.DataFrame([
    ("Source columns including audio", len(source_columns)),
    ("Source metadata columns read", len(READ_COLUMNS)),
    ("Added provenance columns", 2),
    ("Saved metadata columns", len(metadata.columns)),
    ("Source clips — all languages / all splits", len(metadata)),
    ("Thai clips — all splits", len(thai)),
    ("Audio files downloaded/decoded by this workflow", 0),
], columns=["item", "value"])
table("quick_summary", quick)
print("Metadata snapshot size:", round(snapshot.stat().st_size / 1e6, 2), "MB")
print("First fetch:", provenance["fetched_at_utc"])
print("Checksum:", provenance["metadata_sha256"])

## 3. คอลัมน์และความหมาย
นับคอลัมน์ต้นทางแยกจากคอลัมน์ที่เราเพิ่ม `official_split` และ `source_parquet` ไม่ใช่ feature ของโมเดล

`source_model` คือระบบสร้างเสียง ไม่ใช่ AASIST/RawNet2; `speaker_or_voice` อาจเป็นรหัสเสียงสังเคราะห์ ไม่ยืนยันจำนวนคน

In [ ]:
# Data dictionary/schema จริงจาก revision เดียวกัน
dictionary = pd.DataFrame(source_columns)
dictionary["meaning"] = dictionary["column"].map(MEANINGS)
dictionary["selected_for_read"] = dictionary["column"].isin(READ_COLUMNS)
table("source_schema_dictionary", dictionary)
table("added_columns", pd.DataFrame([
    {"column": name, "meaning": MEANINGS[name]}
    for name in ("official_split", "source_parquet")
]))
shards = pd.DataFrame([{key: value for key, value in item.items()
                       if key not in ("source_schema", "read_columns")}
                      for item in provenance["files"]])
table("remote_shard_inventory", shards, limit=40)
scope = pd.DataFrame([
    ("Age column", "absent"),
    ("Gender column", "absent"),
    ("Residence/accent column", "absent"),
    ("Duration column", "absent — hours unavailable in metadata-only scope"),
    ("Audio headers / sample rate verification", "not done"),
    ("Audio decoding / listening / waveform quality", "not done"),
    ("Audio duplicate hashes", "not done"),
    ("Model inference / training / EER", "not done"),
    ("Original local dataset files", "not used"),
], columns=["check", "status"])
table("scope_status", scope)

## 4. Missing data เฉพาะภาษาไทย
ตารางเดียวแสดงทุกคอลัมน์ metadata ต้นทาง 17 คอลัมน์: null แยกจากช่องข้อความว่าง/ช่องว่างล้วน และจำนวนค่าที่ไม่ซ้ำหลังตัด missing

คอลัมน์ที่ไม่มีใน schema เช่น age/gender ไม่ใช่ “มีแต่ missing 100%” ค่าว่างบางช่องอาจไม่ applicable เช่น spoof_type ในเสียงจริง ต้องดูแยกตาม label

In [ ]:
# Missing data — ไม่แสดงตารางกรอง missing ซ้ำกับตารางเต็ม
missing = table("thai_missing_data", missing_profile(thai, READ_COLUMNS))
by_group = []
for (split, label), frame in thai.groupby(["official_split", "label"], dropna=False):
    result = missing_profile(frame, READ_COLUMNS)
    result.insert(0, "label", label)
    result.insert(0, "official_split", split)
    result["group_clips"] = len(frame)
    by_group.append(result)
missing_by_split_label = pd.concat(by_group, ignore_index=True)
table("thai_missing_by_split_label", missing_by_split_label, limit=12)
coverage_rows = []
for label, frame in thai.groupby("label", dropna=False):
    for field in ("speaker_or_voice", "source_dataset", "source_model", "text"):
        count = int((~blank_mask(frame[field])).sum())
        coverage_rows.append({"label": label, "column": field, "clips": len(frame),
                              "nonblank_clips": count, "missing_clips": len(frame) - count,
                              "nonblank_pct": count / len(frame) * 100})
table("thai_metadata_coverage_by_label", pd.DataFrame(coverage_rows))
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(missing["column"][::-1], missing["missing_pct"][::-1])
ax.set(xlabel="Missing (%)", title="Thai source metadata — missing coverage")
save_plot("01_thai_missing")

## 5. ภาพรวมทุกภาษา และจำนวน Train / Dev / Test ของไทย
รักษา split ตามต้นทาง: validation ใช้เป็น Dev, evaluation ใช้เป็น Test ไม่มีการสุ่มแบ่งใหม่

จำนวนคลิปไม่ใช่จำนวนผู้พูด ไม่สรุปชั่วโมงเสียงจาก sample rate หรือจำนวนแถว

In [ ]:
# ภาพรวมภาษาและ split (อ่านใหม่ทั้งหมดจาก HF)
all_counts = metadata.groupby(["official_split", "language", "label"], dropna=False).size().rename("clips").reset_index()
table("all_language_split_label_counts", all_counts, limit=50)
language_counts = metadata.groupby("language", dropna=False).size().rename("clips").reset_index().sort_values("clips", ascending=False)
language_counts["pct_all_clips"] = language_counts["clips"] / len(metadata) * 100
table("all_language_counts", language_counts)
split_counts = thai.groupby(["official_split", "label"], dropna=False).size().unstack(fill_value=0).reindex(SPLITS, fill_value=0)
for label in ("bonafide", "spoof"):
    if label not in split_counts:
        split_counts[label] = 0
split_counts = split_counts[["bonafide", "spoof"]]
split_counts["total"] = split_counts.sum(axis=1)
split_counts["spoof_pct"] = split_counts["spoof"] / split_counts["total"] * 100
table("thai_split_label_counts", split_counts.reset_index())
checks = pd.DataFrame([
    ("Card total at pinned revision", 553746, len(metadata)),
    ("Card Thai total at pinned revision", 85215, len(thai)),
    ("Card Train all languages", 439362, int(metadata["official_split"].eq("train").sum())),
    ("Card Validation all languages", 57158, int(metadata["official_split"].eq("validation").sum())),
    ("Card Evaluation all languages", 57226, int(metadata["official_split"].eq("evaluation").sum())),
], columns=["check", "expected_from_card", "observed"])
checks["matches"] = checks["expected_from_card"].eq(checks["observed"])
table("card_count_crosscheck", checks)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(language_counts["language"], language_counts["clips"])
axes[0].set(title="All languages / all source splits", ylabel="Clips")
split_counts[["bonafide", "spoof"]].plot.bar(stacked=True, ax=axes[1])
axes[1].set(title="Thai clips by official split", ylabel="Clips", xlabel="")
axes[1].tick_params(axis="x", rotation=0)
save_plot("02_language_and_thai_splits")

## 6. แหล่งเสียงจริง/ปลอม และระบบสร้างเสียง
เก็บค่าต้นทาง ไม่เปลี่ยน `none` เป็น missing เอง คำว่า offline/online เป็นหมวดต้นทาง ไม่ใช่การยืนยันว่าเสียงปลอมทุกคลิปเป็น voice cloning

แสดงเพียงค่าหมวดและจำนวน ไม่เปิด transcript รายคลิปโดยค่าเริ่มต้น

In [ ]:
# แจกแจง label / source categories / source_model
for field in ("spoof_type", "category", "source_model", "source_dataset", "mapping_source", "text_granularity"):
    view = thai[["official_split", "label", field]].copy()
    view[field] = view[field].astype("string").str.strip().mask(blank_mask(view[field]), "(missing)")
    counts = view.groupby(["official_split", "label", field], dropna=False).size().rename("clips").reset_index()
    totals = counts.groupby(["official_split", "label"])["clips"].transform("sum")
    counts["pct_within_split_label"] = counts["clips"] / totals * 100
    table(f"thai_{field}_counts", counts.sort_values(["official_split", "label", "clips"], ascending=[True, True, False]), limit=30)
model_mix = thai.loc[thai["label"].eq("spoof")].groupby(["official_split", "source_model"], dropna=False).size().unstack(fill_value=0).reindex(SPLITS)
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(model_mix.fillna(0), ax=ax, annot=True, fmt=".0f", cmap="Blues")
ax.set(title="Thai spoof — generating systems by split", xlabel="source_model", ylabel="")
save_plot("03_thai_generating_systems")

## 7. ความครบถ้วนของข้อความและ flags ต้นทาง
ความยาวที่วัดเป็นจำนวนตัวอักษร ไม่ใช่จำนวนคำ/พยางค์หรือคุณภาพเสียง ไม่ฟังตรวจความตรงของ transcript

`is_text_exact` เป็นคำรับรองระดับ metadata ของต้นทาง จึงต้องอ่านร่วมกับ provenance

In [ ]:
# วิเคราะห์ข้อความโดยไม่แสดงข้อความรายคลิป
thai["normalized_text"] = normalized_text(thai["text"])
text_rows = []
for (split, label), frame in thai.groupby(["official_split", "label"], dropna=False):
    lengths = frame["text"].astype("string").fillna("").str.len()
    nonblank = ~blank_mask(frame["text"])
    valid = lengths[nonblank]
    text_rows.append({"official_split": split, "label": label, "clips": len(frame),
                     "blank_text_clips": int((~nonblank).sum()),
                     "unique_normalized_nonblank_texts": frame.loc[frame["normalized_text"].ne(""), "normalized_text"].nunique(),
                     "median_nonblank_characters": valid.median(),
                     "min_nonblank_characters": valid.min(), "max_nonblank_characters": valid.max()})
table("thai_text_quality_summary", pd.DataFrame(text_rows))
for field in ("is_text_exact", "audio_was_resampled", "sampling_rate"):
    table(f"thai_{field}_metadata_counts",
          thai.groupby(["official_split", "label", field], dropna=False).size().rename("clips").reset_index())
table("thai_text_provenance_counts",
      thai.groupby(["official_split", "label", "mapping_source", "text_granularity", "is_text_exact"], dropna=False).size().rename("clips").reset_index())
fig, ax = plt.subplots(figsize=(9, 4))
for split in SPLITS:
    values = thai.loc[thai["official_split"].eq(split), "text"].astype("string").fillna("").str.len()
    sns.ecdfplot(values, ax=ax, label=split)
ax.set(title="Thai transcript character length", xlabel="Characters", ylabel="Cumulative proportion")
ax.legend()
save_plot("04_thai_text_lengths")

## 8. ข้อมูลผู้พูด/เสียง และ overlap ข้าม split
นับ **ค่ารหัส** ไม่ใช่จำนวนบุคคล โดยเฉพาะเสียงสังเคราะห์

ตรวจทั้งรหัส voice แบบดิบ และ candidate key ที่จับคู่กับ source_dataset/source_model เพื่อไม่รวมรหัสชื่อเดียวกันจากคนละระบบเป็นคนเดียว แต่ยังไม่ยืนยันตัวตน และรหัสที่ขาดหายทำให้พิสูจน์ speaker-disjoint ไม่ได้

In [ ]:
# ความครอบคลุมรหัส และ overlap metadata เท่านั้น
thai["namespaced_voice_key"] = namespaced_voice_keys(thai)
voice_rows = []
for (split, label), frame in thai.groupby(["official_split", "label"], dropna=False):
    known = ~blank_mask(frame["speaker_or_voice"])
    scoped = frame["namespaced_voice_key"].ne("")
    voice_rows.append({"official_split": split, "label": label, "clips": len(frame),
                      "clips_with_raw_voice_code": int(known.sum()),
                      "unique_raw_voice_codes": int(frame.loc[known, "speaker_or_voice"].nunique()),
                      "clips_with_namespaced_candidate_key": int(scoped.sum()),
                      "unique_namespaced_candidate_keys": int(frame.loc[scoped, "namespaced_voice_key"].nunique()),
                      "verified_people": "not available"})
table("thai_voice_code_coverage", pd.DataFrame(voice_rows))
overlaps = pd.concat([overlap_table(thai, field)
                     for field in ("row_id", "utterance_id", "normalized_text", "speaker_or_voice", "namespaced_voice_key")], ignore_index=True)
table("thai_cross_split_metadata_overlap", overlaps, limit=20)
integrity = []
for scope_name, frame in (("all_languages", metadata), ("thai", thai)):
    for field in ("row_id", "utterance_id"):
        nonblank = ~blank_mask(frame[field])
        values = frame.loc[nonblank, field].astype("string").str.strip()
        counts = values.value_counts()
        integrity.append({"scope": scope_name, "column": field,
                          "blank_rows": int((~nonblank).sum()), "unique_nonblank_values": len(counts),
                          "repeated_value_groups": int(counts.gt(1).sum()),
                          "rows_in_repeated_groups": int(counts[counts.gt(1)].sum()),
                          "extra_rows_beyond_first": int((counts - 1).clip(lower=0).sum())})
table("source_id_integrity", pd.DataFrame(integrity))
label_conflicts = []
for field in ("row_id", "utterance_id", "normalized_text"):
    valid = thai.loc[~blank_mask(thai[field])]
    grouped = valid.groupby(field)["label"].nunique()
    label_conflicts.append({"field": field, "values_with_multiple_labels": int(grouped.gt(1).sum()),
                           "interpretation": "same text can legitimately pair real/spoof" if field == "normalized_text" else "inspect source identity; do not auto-delete"})
table("thai_metadata_label_conflicts", pd.DataFrame(label_conflicts))

## 9. สรุปสำหรับ Progress และข้อจำกัด
ผลจาก **metadata ที่อ่านใหม่ครบทั้งสาม split** แยกจากรายงานตรวจเสียงเดิม

- จำนวนคลิป / label / missing / แหล่งเสียง / metadata overlap ตอบได้
- ชั่วโมงเสียง ความยาวคลิปจริง channels คุณภาพเสียง silence/clipping และเสียงซ้ำจาก hash ยังตอบไม่ได้ เพราะไม่อ่าน audio และไม่มี duration ใน schema
- อายุ เพศ จังหวัด/สำเนียงไม่มีใน schema ห้ามเดาจากรหัสหรือเสียง
- ข้อความเดียวกันอาจเป็นคู่เสียงจริง–ปลอม ไม่ถือเป็น audio leakage อัตโนมัติ
- ห้ามอ้างว่ารหัสผู้พูดไม่ซ้ำหรือ ID คลิปไม่ซ้ำ = speaker-disjoint
- ยังไม่ได้ train, inference, เลือก threshold หรือประเมิน EER
- ไม่ลบ/ปรับ split และไม่แก้ข้อมูลต้นทางอัตโนมัติ

การศึกษาความผิดพลาดตามกลุ่มประชากรต้องใช้ชุดเสริมที่มี metadata และรายงานข้อจำกัดของแต่ละชุด

In [ ]:
# บันทึกผลและสรุป โดยไม่เขียนทับรายงานเสียงเดิม
limitations = [
    "First fetch uses remote HF only; original local SEA-Spoof shards/manifests/audio not used.",
    "No audio column selected; no complete source Parquet shards or audio files saved.",
    "No duration column, so audio hours/length unavailable in this metadata-only scope.",
    "No audio header, waveform, hash, listening, SNR, training or inference checks.",
    "Age/gender/residence/accent columns absent; do not impute.",
    "speaker_or_voice and namespaced keys are codes, not verified persons.",
    "Text/ID/code overlaps are not proof of identical audio or speaker identity.",
    "No automatic deduplication, split changes or threshold selection.",
    "Restricted dataset; metadata and outputs must not be publicly redistributed.",
]
summary = {
    "dataset": DATASET, "revision": REVISION, "started_at_utc": STARTED,
    "finished_at_utc": datetime.now(timezone.utc).isoformat(),
    "scope": {"metadata_only": True, "source": "remote_hugging_face",
              "original_local_dataset_files_used": False,
              "audio_column_read": False, "audio_files_downloaded": 0,
              "source_columns_including_audio": len(source_columns),
              "source_metadata_columns_read": len(READ_COLUMNS),
              "added_provenance_columns": 2, "source_shards": provenance["shards"],
              "all_language_clips": len(metadata), "thai_clips": len(thai),
              "splits": list(SPLITS), "audio_hours": None,
              "speaker_disjoint_verified": False},
    "metadata_snapshot_relative_path": str(snapshot.relative_to(ROOT)),
    "metadata_snapshot_bytes": snapshot.stat().st_size,
    "metadata_sha256": provenance["metadata_sha256"],
    "thai_split_label_counts": tables["thai_split_label_counts"].to_dict("records"),
    "thai_missing": missing.to_dict("records"),
    "cross_split_metadata_overlap": overlaps.to_dict("records"),
    "card_count_crosscheck": checks.to_dict("records"),
    "versions": {name: version(name) for name in ["pandas", "pyarrow", "huggingface-hub", "nbformat", "nbconvert"]},
    "limitations": limitations,
}
(OUTPUT / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
display(tables["thai_split_label_counts"])
print(f"SEA-Spoof Thai: {len(thai):,} clips; source {len(source_columns)} columns; read {len(READ_COLUMNS)} metadata columns")
print("Train included; no audio downloaded/decoded; hours unavailable")
print(f"Tables: {len(tables)}; plots: 4; folder: {OUTPUT}")
for limitation in limitations:
    print("-", limitation)